# Machine Learning Lab – Practical 1: Uber Fare Prediction

**Aim:** Predict the fare of an Uber ride using Linear Regression and Random Forest Regression.

**Dataset:** `uber.csv` (Uber Fares Dataset)

## Problem statement
Given pickup and drop-off coordinates and passenger count, build regression models to estimate the ride fare.

## Tasks
1. Load and preprocess the dataset.
2. Identify and remove invalid values and outliers.
3. Calculate the ride distance using the Haversine formula.
4. Study correlations between numerical variables.
5. Train Linear Regression and Random Forest Regression models.
6. Evaluate and compare the models using R², MAE and RMSE.

> Keep `uber.csv` in the same folder as this notebook before running it. This notebook follows your `Linear&RandomForestRegression.ipynb` code as the primary reference.

## 1. Concepts required for this practical

- **Regression:** A supervised machine-learning task used to predict a continuous numerical value. Here, the target is `fare_amount`.
- **Feature (X):** Input information used by a model, such as pickup/drop-off coordinates, passenger count and calculated distance.
- **Target (y):** The value the model must predict; here, Uber fare.
- **Preprocessing:** Cleaning and preparing raw data before training.
- **Outlier:** A value unusually far from the general pattern, such as an implausibly high fare.
- **Feature engineering:** Creating a useful input feature from existing columns. Here, distance is calculated from geographic coordinates.
- **Train/test split:** Keeping some data for training and some unseen data for evaluation.

## 2. Import required libraries

`pandas` manages tabular data, `numpy` performs numerical calculations, `matplotlib` and `seaborn` create graphs, and scikit-learn provides the models, train/test split and evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 3. Load and inspect the dataset

The dataset is loaded into a Pandas DataFrame. `head()` displays the first five records, `shape` gives rows and columns, `info()` shows data types and non-null counts, and `isnull().sum()` counts missing values in each column.

The column-name normalization below handles a possible copied spelling variation in the drop-off coordinate names.

In [ ]:
df = pd.read_csv("uber.csv")

# Normalize possible spelling variations in the drop-off column names.
df.rename(columns={
    "dropoƯ_longitude": "dropoff_longitude",
    "dropoƯ_latitude": "dropoff_latitude"
}, inplace=True)

print("First 5 rows:")
display(df.head())

print("Dataset shape:", df.shape)
print("\\nDataset information:")
df.info()

print("\\nMissing values:")
print(df.isnull().sum())

print("\\nColumn names:")
print(df.columns.tolist())

## 4. Data preprocessing

- Remove `Unnamed: 0` if it exists. It is usually an exported row index, not a useful ride feature.
- Remove duplicate rows.
- Remove rows containing missing values.
- Convert the columns needed for calculations and modelling to numeric values. Any values that cannot be converted become missing (`NaN`) and are removed.

**Why preprocess?** Models need consistent, valid input values. Missing or incorrectly typed values can cause errors or unreliable predictions.

In [ ]:
# Drop an unnecessary exported index column if present.
if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

# Remove duplicate and missing records.
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

# Confirm that all required columns are available.
required_columns = [
    "fare_amount", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "passenger_count"
]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

# Convert modelling columns to numeric.
for col in required_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows made invalid during numeric conversion.
df.dropna(subset=required_columns, inplace=True)

print("Shape after preprocessing:", df.shape)
display(df.head())

## 5. Remove invalid values

This step removes records that cannot represent a valid ride:
- Fare must be greater than zero.
- Passenger count is restricted to 1–6, consistent with the range used in your original practical.
- Latitude must be between −90 and 90 degrees.
- Longitude must be between −180 and 180 degrees.

These coordinate ranges are geographic validity checks; they do not by themselves guarantee that a coordinate is inside the actual service area.

In [ ]:
df = df[df["fare_amount"] > 0].copy()

df = df[
    (df["passenger_count"] >= 1) &
    (df["passenger_count"] <= 6)
].copy()

df = df[
    df["pickup_latitude"].between(-90, 90) &
    df["dropoff_latitude"].between(-90, 90) &
    df["pickup_longitude"].between(-180, 180) &
    df["dropoff_longitude"].between(-180, 180)
].copy()

print("Shape after removing invalid values:", df.shape)

## 6. Calculate ride distance using the Haversine formula

The Haversine formula calculates the great-circle distance between two points on a sphere using their latitude and longitude.

- Latitude and longitude are initially in degrees.
- Trigonometric functions require radians, so coordinates are converted.
- `R = 6371` represents the approximate Earth's radius in kilometres.
- The result is the straight-line surface distance in kilometres, not the actual road distance travelled.

Your original notebook used the same formula with a row-wise `apply`. The version below uses NumPy vector operations, which is generally faster on a large dataset.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in kilometres

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = lat2 - lat1
    dlon = np.radians(lon2) - np.radians(lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    # Clip protects against tiny floating-point errors.
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    return R * c


df["distance_km"] = haversine(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"]
)

# Remove trips with zero calculated distance.
df = df[df["distance_km"] > 0].copy()

print("Distance feature created.")
display(df[["pickup_latitude", "pickup_longitude",
            "dropoff_latitude", "dropoff_longitude",
            "distance_km"]].head())

## 7. Identify and remove outliers using IQR

The Interquartile Range (IQR) method identifies unusually low or high values.

- Q1 = 25th percentile.
- Q3 = 75th percentile.
- IQR = Q3 − Q1.
- Lower limit = Q1 − 1.5 × IQR.
- Upper limit = Q3 + 1.5 × IQR.

Values outside these limits are removed. As in your reference notebook, this is applied to `fare_amount` and `distance_km`. IQR is a statistical rule of thumb; it can remove legitimate extreme rides too, so explain this limitation if asked in viva.

In [ ]:
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    return data[
        (data[column] >= lower_limit) &
        (data[column] <= upper_limit)
    ].copy()


print("Shape before outlier removal:", df.shape)

df = remove_outliers(df, "fare_amount")
df = remove_outliers(df, "distance_km")

print("Shape after outlier removal:", df.shape)

## 8. Correlation analysis

Correlation measures the strength and direction of a linear relationship between numerical variables. Its value ranges from −1 to +1:
- Close to +1: strong positive linear relationship.
- Close to −1: strong negative linear relationship.
- Close to 0: weak or no linear relationship.

The heatmap displays the correlation matrix. In this practical, pay particular attention to how `distance_km` relates to `fare_amount`. Correlation does not prove causation.

In [ ]:
correlation = df[
    [
        "fare_amount", "passenger_count",
        "pickup_longitude", "pickup_latitude",
        "dropoff_longitude", "dropoff_latitude",
        "distance_km"
    ]
].corr()

print("Correlation matrix:")
display(correlation)

plt.figure(figsize=(10, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 9. Select features and target; split the dataset

`X` contains the input features and `y` contains the fare to predict. `train_test_split` separates the data into:
- 80% training data (`test_size=0.20` means 20% is reserved for testing).
- 20% testing data.

`random_state=42` makes the split reproducible. The model is trained only on training data and evaluated on the held-out test data.

In [ ]:
X = df[
    [
        "passenger_count",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "distance_km"
    ]
]

y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training feature rows:", X_train.shape[0])
print("Testing feature rows:", X_test.shape[0])
print("Number of input features:", X_train.shape[1])

## 10. Train the Linear Regression model

Linear Regression estimates a continuous target by fitting a linear relationship between input features and the target.

- `LinearRegression()` creates the model.
- `fit()` learns the relationship from training features and fares.
- `predict()` estimates fares for the test features.

The predicted values are stored in `linear_prediction`.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_prediction = linear_model.predict(X_test)

print("Linear Regression model trained successfully.")

## 11. Train the Random Forest Regression model

Random Forest is an ensemble learning algorithm. It builds multiple decision trees and combines their predictions (averaging for regression).

- `n_estimators=100` creates 100 decision trees.
- `random_state=42` makes the random process reproducible.
- `n_jobs=-1` allows scikit-learn to use available CPU cores.
- `fit()` trains the model; `predict()` predicts fares for test records.

The predictions are stored in `rf_prediction`.

In [ ]:
random_forest = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)
rf_prediction = random_forest.predict(X_test)

print("Random Forest Regression model trained successfully.")

## 12. Evaluate both regression models

Three evaluation metrics are used:

**R² score (coefficient of determination):** Measures how well predictions explain variation in the target. A value closer to 1 is generally better; it can be negative if the model performs worse than predicting the test-set mean.

**MAE (Mean Absolute Error):** Average absolute difference between actual and predicted fares. Lower is better. It is expressed in the same unit as fare.

**RMSE (Root Mean Squared Error):** Square root of the average squared prediction error. Lower is better, and it penalizes large errors more heavily than MAE.

Do not decide the best model from accuracy: this is a regression problem, so use regression metrics.

In [ ]:
def evaluate_model(name, actual, predicted):
    r2 = r2_score(actual, predicted)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    print("\\n" + name)
    print("---------------------------")
    print("R2 Score :", round(r2, 4))
    print("MAE      :", round(mae, 4))
    print("RMSE     :", round(rmse, 4))

    return r2, mae, rmse


linear_scores = evaluate_model(
    "Linear Regression", y_test, linear_prediction
)

rf_scores = evaluate_model(
    "Random Forest Regression", y_test, rf_prediction
)

## 13. Compare model performance

The comparison table places the metrics side by side. A stronger model generally has a higher R² and lower MAE/RMSE. The following code selects the model with the lower RMSE; if RMSE is equal, it reports a tie for this comparison.

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regression"],
    "R2 Score": [linear_scores[0], rf_scores[0]],
    "MAE": [linear_scores[1], rf_scores[1]],
    "RMSE": [linear_scores[2], rf_scores[2]]
})

print("Model Comparison:")
display(results)

if rf_scores[2] < linear_scores[2]:
    print("Random Forest Regression has the lower RMSE.")
elif linear_scores[2] < rf_scores[2]:
    print("Linear Regression has the lower RMSE.")
else:
    print("Both models have the same RMSE.")

## 14. Actual vs Predicted fare plot

This scatter plot compares actual test fares with Random Forest predictions. If predictions were perfect, points would lie on the diagonal line `y = x`. Points far from that line indicate larger prediction errors.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, rf_prediction, alpha=0.5)

# Reference line: perfect predictions.
min_value = min(y_test.min(), rf_prediction.min())
max_value = max(y_test.max(), rf_prediction.max())
plt.plot([min_value, max_value], [min_value, max_value], "r--")

plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Actual vs Predicted Fare - Random Forest")
plt.tight_layout()
plt.show()

## 15. Conclusion

In this practical, the Uber fare dataset is cleaned and prepared, invalid records and outliers are handled, and ride distance is calculated using the Haversine formula. Correlation analysis is performed, then Linear Regression and Random Forest Regression are trained to predict fare amounts. Their performance is compared using R², MAE and RMSE.

**Write the result using your own execution output:** State which model performed better according to the metric(s) observed in your run. Do not copy assumed scores because results depend on the dataset and environment.

## 16. Important viva questions

1. **What is machine learning?**  
   Machine learning is a field of AI in which systems learn patterns from data to make predictions or decisions.

2. **What is regression?**  
   Regression predicts a continuous numerical value, such as fare amount.

3. **Why is this a regression problem and not classification?**  
   Because the output is a numerical fare, not a category or class.

4. **What is the target variable?**  
   `fare_amount`.

5. **What are the features used in this practical?**  
   Passenger count, pickup and drop-off latitude/longitude, and calculated distance.

6. **What is data preprocessing?**  
   Cleaning and transforming raw data so it can be used reliably by a machine-learning model.

7. **What is an outlier?**  
   A value that lies unusually far from the general distribution of the data.

8. **What is IQR?**  
   IQR is the difference between the third quartile and first quartile: Q3 − Q1.

9. **What is the Haversine formula used for?**  
   It calculates the great-circle distance between two points on a sphere using latitude and longitude.

10. **Why convert degrees into radians?**  
    NumPy trigonometric functions expect angles in radians.

11. **What is correlation?**  
    A measure of the direction and strength of a relationship between two variables.

12. **What is Linear Regression?**  
    A regression algorithm that models a linear relationship between input features and a numerical target.

13. **What is Random Forest Regression?**  
    An ensemble method that combines predictions from multiple decision trees.

14. **What is the purpose of train_test_split?**  
    It separates data into training and testing subsets to evaluate performance on unseen records.

15. **What does `random_state=42` do?**  
    It makes the random split/model process reproducible.

16. **What is R²?**  
    A metric that measures how much of the variation in the target is explained by the model.

17. **What is MAE?**  
    The average absolute difference between actual and predicted values.

18. **What is RMSE?**  
    The square root of the mean squared prediction error; it penalizes large errors more strongly.

19. **Which is better: lower MAE/RMSE or higher R²?**  
    Generally, lower MAE and RMSE and higher R² indicate better performance.

20. **Why might Random Forest perform differently from Linear Regression?**  
    Random Forest can model nonlinear patterns and feature interactions, while ordinary Linear Regression models a linear relationship.

## 17. Execution checklist and notes

- Place `uber.csv` in the same folder as this notebook, or change the path in the loading cell.
- Run the notebook from the first cell to the last cell in order.
- If a column-name error occurs, compare `df.columns.tolist()` with the required column names.
- If the file is not found, check the notebook's working directory and dataset filename.
- This notebook has been prepared from your reference code, but it has not been run against your local `uber.csv` here. Actual output values and model scores will appear only when you execute it with your dataset.